# Aging PBMCs — compare cell-level Wilcoxon markers, pseudobulk PyDESeq2, and GSEApy

## Problem

How does the method used to define differential genes affect the result, and which PBMC cell types show the largest transcriptomic differences between younger and older healthy adults?

You will analyze an **eight-donor, sex-balanced, age-extreme subset** of the public Zhu et al. human PBMC study (GEO **GSE213516**). The four **young** donors are ages 28–36 and the four **old** donors are ages 60–77. The intermediate ages are deliberately excluded, so the age contrast is explicit rather than arbitrary. The source provides Cell Ranger integer UMI counts and author-provided fine cell-type labels.

This is an exploratory young-versus-old observational comparison. It is **not** a longitudinal aging experiment and cannot establish that age causes an observed expression difference.

## Launch in Google Colab

This file is Colab-ready. Download it, open [Google Colab](https://colab.research.google.com/), and choose **File → Upload notebook**. The first run downloads approximately 0.9 GB of public source data and prepares the eight selected donors; use a Colab runtime with enough disk space and allow extra time for the initial download.

A public **Open in Colab** button is intentionally not shown: this file has not been published to a public, student-safe GitHub repository. Do not add a launch button until the notebook is hosted at a verified public repository path.

## What you will learn

By completing the three sections, you will be able to:

1. **Use the right unit of replication.** Explain why a cell-level Wilcoxon test is useful for exploratory marker ranking but does not make cells independent biological replicates. Construct pseudobulk profiles so **donors**, not cells, are the observations for PyDESeq2.
2. **Read core DEG statistics.** Interpret an effect direction, `log2FoldChange`, ordinary p value, Benjamini–Hochberg adjusted p value / FDR, and the PyDESeq2 Wald statistic. Translate a two-fold threshold into `|log2FoldChange| ≥ 1`.
3. **Create and interpret method-comparison figures.** Make signed-effect correlation plots and **Venn diagrams** (not “Ben diagrams”) to compare the DEG sets reported by Wilcoxon and pseudobulk PyDESeq2.
4. **Test the aging contrast within each cell type.** Count age-associated genes at `padj < 0.01` and `|log2FoldChange| ≥ 1`, identify the two cell types with the largest counts, and interpret a volcano plot without equating significance with biological importance.
5. **Run pathway analysis with GSEApy.** Build a ranked gene list from the DESeq2 Wald statistic, run pre-ranked GSEA against current human **KEGG** and **Gene Ontology Biological Process** resources, and interpret ES, NES, FDR q value, and leading-edge genes.
6. **Write a bounded biological claim.** Separate an observed pattern, an inference, a limitation, and a next validation test.

## What is supplied versus what you must build

The notebook supplies only the environment and **Scanpy data setup**: verified public download, raw-count preservation, author labels, normalized expression, a UMAP for inspection, and a donor-by-cell-type coverage table.

After setup, there is deliberately **no DEG, pseudobulk, PyDESeq2, correlation, Venn, volcano, or GSEApy analysis code**. You will use the staged questions and the AI coaching contract to request one small next step, run it, inspect its diagnostic, and decide what to do next.


## Visual prerequisite map

```text
Prior knowledge
  AnnData • sparse counts • PBMC cell types • basic Python / tables / plots
                                  │
                                  ▼
Data setup supplied here
  public UMI counts → author labels → raw-count layer + log-normalized Scanpy object
                                  │
                 ┌────────────────┴────────────────┐
                 ▼                                 ▼
Section 1 — method comparison            Section 2 — aging DEG
Wilcoxon cell markers versus             young vs old within each cell type
pseudobulk target-versus-rest            donor pseudobulks + PyDESeq2
                 │                                 │
                 └────────────────┬────────────────┘
                                  ▼
Section 3 — pathway inference
ranked GSEApy with KEGG + GO Biological Process
                                  │
                                  ▼
Evidence memo
figures + statistics + limitation + next test
```

**Prerequisites used:** a cell is an observation nested in a donor; raw integer counts differ from log-normalized values; broad PBMC cell types; tables and plots.

**Introduced before analysis:** pseudoreplication, pseudobulk, adjusted p value/FDR, log2 fold change, method agreement, volcano plots, and pre-ranked GSEA.

**Reserved for later:** continuous-age models, donor random-effect models, compositional analysis, trajectory inference, and causal claims about aging.


## Data card and analysis boundaries

| Item | This lab |
|---|---|
| Study | Zhu et al., *Science Advances* 2023; human PBMC single-cell aging study |
| GEO source | GSE213516 Cell Ranger UMI count triplets |
| Selected donors | `F30`, `F36`, `M28`, `M36` (young) and `F66`, `F77`, `M60`, `M66` (old) |
| Young / old rule | Young ages 28–36; old ages 60–77; four donors per group and sex-balanced |
| Count matrix | Integer UMI counts, stored as `adata.layers["counts"]` after setup |
| Cell labels | Author-provided `cell_type` labels from the public study annotation table |
| Wilcoxon role | Exploratory cell-level target-versus-rest marker ranking |
| PyDESeq2 role | Sample-aware pseudobulk contrasts; donors are replicates |
| Primary aging contrast | `Old` versus `Young` within each eligible cell type |
| DEG threshold for Section 2 | `padj < 0.01` and `abs(log2FoldChange) >= 1` |
| Pathway approach | GSEApy pre-ranked GSEA using the DESeq2 Wald statistic, KEGG and GO Biological Process |

**Critical boundary:** the Section 1 PyDESeq2 “target cell type versus pooled rest” comparison is a teaching comparison with Scanpy’s target-versus-rest marker definition. The rest compartment is biologically heterogeneous. Treat it as a method-comparison artifact, not as a unique or causal cell-type signature. The more defensible biological inference in this lab is the Section 2 donor-level young-versus-old contrast within a pre-specified cell type.


In [ ]:
# Install the Colab environment. Run once; restart the runtime if Colab asks.
%pip install --quiet "scanpy>=1.11,<1.13" "anndata>=0.10" "decoupler==2.2.0" "pydeseq2==0.5.4" "gseapy>=1.1" "matplotlib-venn>=1.1" "adjustText>=1.3" "seaborn>=0.13"


In [ ]:
# Imports, reproducibility settings, and output folder.
# This is setup code only. The DEG and pathway-analysis code is intentionally not supplied later.
import json
import os
import random
import tarfile
import urllib.request
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

WORKDIR = Path("aging_pbmc_gse213516")
MTX_DIR = WORKDIR / "selected_matrices"
OUTPUT_DIR = Path("outputs_aging_pbmc")
WORKDIR.mkdir(exist_ok=True)
MTX_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Scanpy:", sc.__version__)
print("AnnData:", ad.__version__)
print("Working directory:", Path.cwd().resolve())


In [ ]:
# Local, device-only learning checklist. This is not a gradebook and stores no identity.
PROGRESS = {
    "setup_audited": False,
    "method_comparison": False,
    "aging_deg": False,
    "pathway_gsea": False,
    "evidence_memo": False,
}
PROGRESS_FILE = Path("aging_pbmc_progress.json")

def save_progress(path=PROGRESS_FILE):
    payload = {"version": 1, "module": "aging-pbmc-deg-method-comparison", "progress": PROGRESS}
    path.write_text(json.dumps(payload, indent=2) + "\n")
    print(f"Saved portable local progress to: {path.resolve()}")

def load_progress(path=PROGRESS_FILE):
    payload = json.loads(path.read_text())
    if payload.get("version") != 1 or set(payload.get("progress", {})) != set(PROGRESS):
        raise ValueError("This is not a compatible Aging PBMC progress file.")
    PROGRESS.update({key: bool(value) for key, value in payload["progress"].items()})
    print(f"Loaded local progress from: {path.resolve()}")
    print_progress()

def mark_done(*steps):
    for step in steps:
        if step not in PROGRESS:
            raise KeyError(f"Unknown step: {step}. Choose from {list(PROGRESS)}")
        PROGRESS[step] = True
    save_progress()
    print_progress()

def print_progress():
    completed = sum(PROGRESS.values())
    for step, done in PROGRESS.items():
        print(f"{'✓' if done else '○'} {step}")
    print(f"Progress: {completed}/{len(PROGRESS)} — local checklist only, not a grade submission.")

print_progress()
print("Use `save_progress()` to export this device-local checklist, or `load_progress()` after uploading a compatible JSON file.")


In [ ]:
# Visual learning route. It creates no analysis result and can be rerun safely.
from matplotlib.patches import FancyBboxPatch

stages = [
    ("Prerequisites", "AnnData · raw counts\nPBMC types · plots"),
    ("Setup", "Scanpy object\nlabels · count layer"),
    ("1. Compare", "Wilcoxon ↔\npseudobulk PyDESeq2"),
    ("2. Aging DEG", "Old vs Young\nwithin cell type"),
    ("3. Pathways", "GSEApy\nKEGG + GO BP"),
    ("Evidence memo", "claim · limit\nnext test"),
]
fig, ax = plt.subplots(figsize=(15, 2.8))
ax.set_axis_off()
for i, (title, body) in enumerate(stages):
    x = 0.02 + i * 0.163
    box = FancyBboxPatch((x, 0.25), 0.135, 0.5, boxstyle="round,pad=0.02", facecolor="#F4F7FA", edgecolor="#008D8D", linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 0.0675, 0.58, title, ha="center", va="center", fontsize=11, weight="bold", color="#0A192F")
    ax.text(x + 0.0675, 0.39, body, ha="center", va="center", fontsize=9, color="#4A5568")
    if i < len(stages) - 1:
        ax.annotate("", xy=(x + 0.16, 0.5), xytext=(x + 0.14, 0.5), arrowprops={"arrowstyle": "->", "color": "#008D8D", "lw": 2})
plt.show()


# Setup — public data download and Scanpy object

The next cells are the supplied setup. They download the official GEO archive once, extract only the selected eight donors, attach the public author labels by barcode, preserve raw UMI counts, and create a normalized Scanpy object for visualization and Wilcoxon ranking.

The original matrix values are integer UMI counts. After normalization, `.X` becomes log-normalized expression for visualization and cell-level Wilcoxon ranking. The untouched raw counts remain in `adata.layers["counts"]` for all pseudobulk and PyDESeq2 work. Do not use log-normalized `.X` as a PyDESeq2 count matrix.


In [ ]:
# Pre-specified, sex-balanced age-extreme teaching subset.
# Intermediate-age donors are excluded rather than silently assigned to young or old.
SAMPLES = pd.DataFrame([
    {"gsm": "GSM6588511", "sample_id": "F30", "age": 30, "sex": "Female", "age_group": "Young"},
    {"gsm": "GSM6588512", "sample_id": "F36", "age": 36, "sex": "Female", "age_group": "Young"},
    {"gsm": "GSM6588522", "sample_id": "M28", "age": 28, "sex": "Male", "age_group": "Young"},
    {"gsm": "GSM6588523", "sample_id": "M36", "age": 36, "sex": "Male", "age_group": "Young"},
    {"gsm": "GSM6588515", "sample_id": "F66", "age": 66, "sex": "Female", "age_group": "Old"},
    {"gsm": "GSM6588516", "sample_id": "F77", "age": 77, "sex": "Female", "age_group": "Old"},
    {"gsm": "GSM6588525", "sample_id": "M60", "age": 60, "sex": "Male", "age_group": "Old"},
    {"gsm": "GSM6588526", "sample_id": "M66", "age": 66, "sex": "Male", "age_group": "Old"},
])
SAMPLES["age_group"] = pd.Categorical(SAMPLES["age_group"], categories=["Young", "Old"], ordered=True)
display(SAMPLES)
assert SAMPLES.groupby("age_group", observed=True)["sample_id"].nunique().to_dict() == {"Young": 4, "Old": 4}
assert SAMPLES.groupby(["age_group", "sex"], observed=True).size().to_dict() == {
    ("Young", "Female"): 2, ("Young", "Male"): 2,
    ("Old", "Female"): 2, ("Old", "Male"): 2,
}


In [ ]:
# Download the official GEO archive and public author annotation table.
# This archive is approximately 0.9 GB; only the 24 selected 10x matrix files are extracted below.
GEO_TAR_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213516/suppl/GSE213516_RAW.tar"
ANNOTATION_URL = "https://ndownloader.figshare.com/files/41726226"
GEO_TAR = WORKDIR / "GSE213516_RAW.tar"
ANNOTATION_CSV = WORKDIR / "SE_CYCT_JOCT_WHCT_SC_meta.csv"

def download_if_missing(url, destination):
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Using cached: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"Downloading: {destination.name}")
    urllib.request.urlretrieve(url, destination)
    print(f"Saved: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")

download_if_missing(GEO_TAR_URL, GEO_TAR)
download_if_missing(ANNOTATION_URL, ANNOTATION_CSV)


In [ ]:
# Extract only the selected donors' 10x barcode, feature, and UMI-count matrix files.
# Scanpy reads the resulting gzipped triplets directly through the filename prefix.
selected_prefixes = [f"{row.gsm}_{row.sample_id}_" for row in SAMPLES.itertuples(index=False)]
expected_names = {
    f"{prefix}{suffix}"
    for prefix in selected_prefixes
    for suffix in ["barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz"]
}

missing = [name for name in expected_names if not (MTX_DIR / name).exists()]
if missing:
    with tarfile.open(GEO_TAR, "r") as archive:
        available = {Path(member.name).name: member for member in archive.getmembers()}
        unresolved = sorted(set(missing) - set(available))
        if unresolved:
            raise FileNotFoundError(f"Selected archive members not found: {unresolved[:5]}")
        for name in missing:
            member = available[name]
            with archive.extractfile(member) as source, open(MTX_DIR / name, "wb") as destination:
                destination.write(source.read())
    print(f"Extracted {len(missing)} selected files.")
else:
    print("All selected matrix files are already extracted.")

assert {path.name for path in MTX_DIR.iterdir()} >= expected_names
print("Selected matrix directory:", MTX_DIR.resolve())


In [ ]:
# Read one sparse 10x matrix per donor, join public author labels by barcode, then concatenate.
# The inner join retains the authors' QC-passing, annotated cells. It is not a positional join.
annotation = pd.read_csv(ANNOTATION_CSV)
required_annotation_columns = {"X", "Age", "sampleID", "Cohort", "Celltype"}
assert required_annotation_columns.issubset(annotation.columns), annotation.columns.tolist()

annotation = annotation.loc[
    annotation["Cohort"].eq("SE") & annotation["sampleID"].isin(SAMPLES["sample_id"]),
    ["X", "Age", "sampleID", "Celltype"],
].copy()
assert annotation["X"].is_unique, "Expected one author label per sample-barcode key."
label_lookup = annotation.set_index("X")["Celltype"]

adatas = []
join_summary = []
for row in SAMPLES.itertuples(index=False):
    prefix = f"{row.gsm}_{row.sample_id}_"
    donor = sc.read_10x_mtx(
        MTX_DIR,
        prefix=prefix,
        var_names="gene_symbols",
        make_unique=True,
        gex_only=True,
        cache=False,
    )
    donor.obs["sample_id"] = row.sample_id
    donor.obs["age"] = row.age
    donor.obs["sex"] = row.sex
    donor.obs["age_group"] = row.age_group
    donor.obs["barcode_no_suffix"] = donor.obs_names.astype(str).str.replace(r"-1$", "", regex=True)
    donor.obs["annotation_key"] = row.sample_id + "_" + donor.obs["barcode_no_suffix"]
    donor.obs["cell_type"] = donor.obs["annotation_key"].map(label_lookup)
    n_before = donor.n_obs
    donor = donor[donor.obs["cell_type"].notna()].copy()
    join_summary.append({"sample_id": row.sample_id, "age_group": row.age_group, "raw_barcodes": n_before, "annotated_cells": donor.n_obs})
    adatas.append(donor)

join_summary = pd.DataFrame(join_summary)
display(join_summary)
assert (join_summary["annotated_cells"] > 0).all(), "Every selected donor should retain author-annotated cells."

adata = ad.concat(adatas, join="inner", index_unique="__")
adata.obs["age_group"] = pd.Categorical(adata.obs["age_group"], categories=["Young", "Old"], ordered=True)
adata.layers["counts"] = adata.X.copy()  # raw Cell Ranger integer UMI counts
raw_values = adata.layers["counts"].data if hasattr(adata.layers["counts"], "data") else np.ravel(adata.layers["counts"])
assert np.allclose(raw_values, np.round(raw_values)), "Expected integer-like UMI counts in the counts layer."
print(adata)
print("Author cell labels:", adata.obs["cell_type"].nunique())


In [ ]:
# Scanpy setup for inspection and cell-level Wilcoxon ranking.
# Keep `adata.layers['counts']` unchanged for later pseudobulk / PyDESeq2 steps.
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)

# `.X` becomes normalized and log-transformed only for visualization and Wilcoxon analysis.
sc.pp.normalize_total(adata, target_sum=10_000)
sc.pp.log1p(adata)
adata.raw = adata  # all log-normalized genes for Scanpy marker ranking

# Use a separate HVG object for the neighborhood graph, leaving main `adata` full-gene.
sc.pp.highly_variable_genes(adata, n_top_genes=2_000, flavor="seurat", batch_key="sample_id")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30, random_state=SEED)
sc.tl.umap(adata_hvg, random_state=SEED)

# Copy only the embedding/graph results back; full-gene `adata` retains the raw count layer.
adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"].copy()
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"].copy()
adata.obsp["connectivities"] = adata_hvg.obsp["connectivities"].copy()
adata.obsp["distances"] = adata_hvg.obsp["distances"].copy()

print("Setup complete.")
print("Use `adata.raw` or log-normalized `.X` for Wilcoxon marker ranking.")
print("Use `adata.layers['counts']` only for pseudobulk / PyDESeq2.")


In [ ]:
# Setup diagnostics: inspect labels, donor balance, and cell-type coverage before any DEG.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.umap(adata, color="age_group", ax=axes[0], show=False, title="Age group")
sc.pl.umap(adata, color="cell_type", ax=axes[1], show=False, title="Author cell-type labels", legend_loc="on data")
plt.tight_layout()
plt.show()

MIN_CELLS_PER_DONOR_CELLTYPE = 20
celltype_sample_counts = (
    adata.obs.groupby(["cell_type", "sample_id", "age_group"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)
coverage = (
    celltype_sample_counts.assign(pass_min_cells=lambda d: d["n_cells"] >= MIN_CELLS_PER_DONOR_CELLTYPE)
    .groupby(["cell_type", "age_group"], observed=True)["pass_min_cells"]
    .sum()
    .unstack(fill_value=0)
    .rename(columns={"Young": "young_donors_passing", "Old": "old_donors_passing"})
)
coverage["eligible_for_aging_pseudobulk"] = (
    (coverage.get("young_donors_passing", 0) >= 3) & (coverage.get("old_donors_passing", 0) >= 3)
)
coverage = coverage.sort_values(["eligible_for_aging_pseudobulk", "young_donors_passing", "old_donors_passing"], ascending=[False, False, False])
display(coverage)

eligible_cell_types = coverage.index[coverage["eligible_for_aging_pseudobulk"]].tolist()
print("Eligible cell types (at least 3 donor pseudobulks per age group at the stated cell-count screen):")
print(eligible_cell_types)
mark_done("setup_audited")


# Stop: from this point forward, you build the analysis

Do **not** search online for a complete solution. The remaining notebook contains questions, required artifacts, and prompts for a staged AI tutor. It intentionally provides no analysis code.

Before asking an AI assistant for code, inspect the diagnostic produced by the previous step. When the assistant suggests a short code fragment, run it yourself and record what you accepted, rejected, or changed. Never paste protected, identifiable, unpublished, embargoed, or access-controlled data into an AI system; this public teaching dataset is suitable for the prompts below.


## AI coaching contract — paste this before asking for help

```text
You are my Socratic single-cell analysis tutor for a public PBMC aging lab. I already have:
- `adata`: a Scanpy AnnData object with `adata.layers["counts"]` holding integer UMI counts;
- log-normalized `.X` and `adata.raw` for cell-level marker ranking;
- `sample_id`, `age_group` (Young / Old), `sex`, and author-provided `cell_type` columns;
- `coverage` and `eligible_cell_types` from the setup diagnostic.

Non-negotiable rules:
1. Do not give a complete notebook, all three sections, a complete loop over cell types, or an answer key.
2. Work on exactly ONE micro-step that I select. Give only the smallest code fragment needed for that micro-step, then stop.
3. Before code, explain the statistical purpose in no more than two short paragraphs. Define any new statistic or plot axis.
4. Tell me one diagnostic to inspect after I run the fragment and one common failure mode.
5. Never invent genes, cell types, numerical results, library names, or biological conclusions. Ask to see my actual diagnostic/table if needed.
6. Distinguish cell-level exploratory Wilcoxon ranking from donor-level pseudobulk PyDESeq2 inference. Do not treat cells as independent replicates.
7. At the end, ask exactly one multiple-choice checkpoint and then offer 3–4 options for the next micro-step. Wait for my choice.
8. Do not move to the next micro-step until I report the result or explicitly choose to continue.

Start by asking me which section and micro-step I choose.
```


## Optional Colab AI prompt studio

Use these only after pasting the AI coaching contract. **Privacy boundary:** never send protected, identifiable, unpublished, embargoed, or access-controlled data to an AI assistant. This public PBMC dataset is suitable, but no AI output replaces rerunning your code and checking your diagnostics.

### Prompt 1 — Explain the decision

```text
Module: Aging PBMC method comparison and age-associated DEG.
Question: Which genes and pathways differ between young and old donors within a cell type, and how does that differ from target-versus-rest marker ranking?
Decision: use log-normalized cell-level data only for exploratory Wilcoxon markers; use donor-level raw-count pseudobulks for PyDESeq2.
Public dataset: GSE213516 selected eight-donor PBMC subset.

Explain why this decision matches the biological unit of replication. Name assumptions and failure modes. Do not write code, analyze results, or replace the staged workflow. End by asking me which one micro-step I want next.
```

### Prompt 2 — Diagnose before editing

```text
Here is a minimal non-identifying traceback or diagnostic summary from this public PBMC lab:
<<PASTE SUMMARY>>

Give three ranked hypotheses, one discriminating check for each, and the smallest reversible change. Preserve raw counts, the current seed, provenance, baseline outputs, and contrast direction. Do not write a complete solution. End with a single multiple-choice question about my next diagnostic.
```

### Prompt 3 — Design one bounded variation

```text
Challenge: change exactly one defensible parameter or design decision in the aging PBMC analysis, such as a cell-coverage screen, gene filter, or age-model covariate.

Propose only one variation. Predict the measurable diagnostic effect, provide only a compact patch if I request code, and state what result would make me retain, narrow, or withdraw an interpretation. Do not run all sections or invent results. End by asking which diagnostic I will check.
```

### Prompt 4 — Audit a claim

```text
Evidence artifact: donor/cell-type coverage table; method-comparison figures; age-DE summary; volcano plots; KEGG and GO GSEA outputs.
Claim: <<PASTE MY CLAIM>>

Separate direct observation, inference, limitation, and next validation test. Flag unsupported causal language, cells-as-replicates reasoning, ambiguous contrast direction, unverified gene names, and a pathway-as-mechanism claim. Do not invent evidence. End with one multiple-choice revision question.
```

**Verification record:** for every AI interaction, record the prompt purpose, what you accepted or rejected, the diagnostic you reran, and whether your claim was retained, narrowed, or withdrawn.


# Section 1 — compare marker genes from Wilcoxon and pseudobulk PyDESeq2

## Goal

For each well-represented author-labeled cell type, compare a **cell-level Wilcoxon target-versus-rest marker ranking** with a **donor-level pseudobulk PyDESeq2 target-versus-pooled-rest contrast**. Quantify agreement in signed effect sizes and overlap in thresholded DEG sets.

This section does **not** ask which cell type changes with age. It asks how two analysis strategies behave when asked to characterize a cell type. Keep the biological limitation visible: “rest” pools heterogeneous cells and is an operational reference, not a homogeneous biological condition.

## Statistics and figures you must understand

- **Wilcoxon rank-sum score:** an ordering statistic for a cell-level group versus rest. Its adjusted p value can be highly optimistic when cells from the same donor are treated as independent.
- **PyDESeq2 `log2FoldChange`:** a signed effect size from a negative-binomial pseudobulk model. Positive means higher in the tested target relative to its stated reference.
- **Wald statistic:** effect divided by its estimated uncertainty; use it for ranking only when the comparison and model are stated.
- **Adjusted p value / FDR:** a multiple-testing correction. It is not the probability that the gene is false.
- **Correlation plot:** compare matched, signed log2 fold changes; use a rank correlation such as Spearman, not a correlation of p values.
- **Venn diagram:** compare thresholded gene **sets**. It cannot show effect direction or effect-size agreement.

Use the same pre-registered comparison rule for both methods: `padj < 0.01` and `abs(log2 fold change) >= 1`, where `abs(log2 fold change) >= 1` means at least a two-fold difference in either direction. Report that Scanpy’s Wilcoxon log-fold change is an approximate mean-log expression contrast, so it is not numerically identical to the PyDESeq2 estimate.


## Section 1 questions and required artifacts

**1A — Define one fair comparison.** Which cell types pass the donor-coverage screen? State which label you will test first and why you should not begin with a rare type. What does “target versus rest” mean in each method?

**1B — Wilcoxon marker ranking.** Run Scanpy Wilcoxon target-versus-rest ranking on log-normalized expression for one cell type. Save a tidy table with gene, signed log-fold change, p value, adjusted p value, and—if available—fraction of expressing cells. Make one readable marker table or dot/violin plot. Explain what an adjusted p value means and why it does not turn individual cells into biological replicates.

**1C — Pseudobulk PyDESeq2 marker contrast.** From `adata.layers["counts"]`, sum counts per donor separately for the target label and the pooled rest. Screen pseudobulk quality and test target versus pooled rest using a design that blocks on donor. Save the PyDESeq2 result table. Explain why raw integer counts are required and why the pooled rest is a limitation.

**1D — Agreement and difference.** Merge the two result tables on gene name. For the union of genes passing the stated DEG rule in either method, make a scatter plot of signed effect sizes with a `y=x` guide and report Spearman correlation. Do not correlate p values. Then make a Venn diagram from the two thresholded sets and a summary table with Wilcoxon-only, PyDESeq2-only, shared, total-per-method, and Jaccard overlap. Repeat only after a one-cell-type pilot passes its diagnostics.

**Deliverable:** a per-cell-type method-comparison table plus correlation plot(s) and Venn diagram(s). Conclude which method calls more genes under the chosen thresholds and why “more” is not automatically “better.”


### Section 1 staged AI prompt

```text
I am working on Section 1 of the Aging PBMC lab. Follow the AI coaching contract exactly.

My goal: compare cell-level Wilcoxon target-versus-rest markers against donor-level pseudobulk PyDESeq2 target-versus-pooled-rest results for one author-labeled PBMC cell type, then repeat only after the pilot works.

My fixed thresholds for the final set comparison are padj < 0.01 and abs(log2 fold change) >= 1. I will compare signed effect sizes with Spearman correlation and gene-set overlap with a Venn diagram. I will not correlate p values.

Teach only the micro-step I choose:
A. Read `coverage` and choose a well-represented pilot cell type; explain the target-versus-rest estimand.
B. Run and export one Scanpy Wilcoxon target-versus-rest result; explain score, p value, adjusted p value, approximate log-fold change, and expression fraction.
C. Construct donor-level raw-count target and pooled-rest pseudobulks; diagnose cell and count coverage before modeling.
D. Fit one paired/blocking PyDESeq2 target-versus-rest model and export a result table; explain the design and contrast.
E. Merge one cell type’s two tables, create the signed-effect correlation plot, calculate Spearman correlation, and explain the axes.
F. Define comparable DEG sets, make one Venn diagram plus summary counts, and explain what the diagram cannot show.
G. Turn the successful pilot into a conservative loop across eligible cell types, with skip/report rules for failed coverage.

First ask me to select A–G. Do not give any later step until I choose it and report the diagnostic.
```


# Section 2 — find age-associated DEG within each cell type

## Goal

Test **Old versus Young** within each eligible cell type using donor-level pseudobulk raw counts and PyDESeq2. Count genes passing the pre-specified threshold, identify the two cell types with the largest number of age-associated genes, and create a volcano plot for each.

This is the main biological comparison. It differs from Section 1 because the condition is age group and the aggregation is **donor × cell type**, not target cell type versus pooled rest.

## Statistics and figures you must understand

- **Design / contrast:** write exactly what the model adjusts for and the direction of the contrast, for example Old relative to Young. Check whether adding sex is estimable and justified with four donors per group; do not add covariates mechanically.
- **Adjusted p value threshold:** `padj < 0.01` controls the expected false-discovery proportion under the model assumptions across the tested genes.
- **Effect-size threshold:** `abs(log2FoldChange) >= 1` is a minimum two-fold difference. It prevents a tiny but very precise effect from qualifying solely because its p value is small.
- **DEG-count plot/table:** ranks cell types by number of thresholded genes. It depends on cell number, donor coverage, filtering, and power; it is not a direct measure of which cell type “ages the most.”
- **Volcano plot:** each dot is one gene; x-axis is signed log2 fold change (Old / Young), y-axis is `−log10(padj)`. Label only selected genes after verifying their names and table values.

Use a separate gene filter inside each cell type before its DESeq2 model. Preserve enough donor pseudobulks in both groups, and report cell types you exclude with the reason.


## Section 2 questions and required artifacts

**2A — Pseudobulk design audit.** Create one raw-count pseudobulk per donor × cell type. For every cell type, tabulate the number of donor profiles and contributing cells by age group. Which cell types remain eligible after your declared sample and cell-count screen? Explain why cell count is not the DESeq2 replicate count.

**2B — One age-DE pilot.** Choose one eligible cell type. Filter lowly expressed genes within that cell type, fit its PyDESeq2 model, and state the contrast direction in words. Examine whether the age-group and sex metadata support the design you intend to use. Treat this small four-versus-four study as exploratory.

**2C — Complete the comparison responsibly.** Apply the same documented filtering, design, contrast, and thresholds across eligible cell types. Build a results-summary table containing number of tested genes, number of genes higher in Old, number higher in Young, total DEG count, and donor coverage.

**2D — Rank cell types without overclaiming.** Make a sorted bar plot of total DEG count. Which two cell types have the greatest counts using `padj < 0.01` and `abs(log2FoldChange) >= 1`? Write one reason this ranking could change with different donor coverage or gene filtering.

**2E — Volcano plots.** For the top two cell types only, make one volcano plot each. Draw threshold guides, distinguish higher-in-Old and higher-in-Young genes, label a small number of verified genes, and provide a caption that states the cell type, contrast, filters, and thresholds.

**Deliverable:** a reproducible cell-type summary table, one DEG-count bar plot, two volcano plots, and a short paragraph that separates an observed pattern from an aging-causality claim.


### Section 2 staged AI prompt

```text
I am working on Section 2 of the Aging PBMC lab. Follow the AI coaching contract exactly.

My goal: test Old versus Young within each eligible author-labeled PBMC cell type using donor-level pseudobulk raw UMI counts and PyDESeq2. I will call DEGs only when padj < 0.01 and abs(log2FoldChange) >= 1. A positive log2 fold change must mean Old / Young in my final report.

Teach only the micro-step I choose:
A. Build donor × cell-type pseudobulks from `adata.layers["counts"]`; inspect donor, cell, and total-count coverage.
B. Decide an age-DE design after inspecting sex balance and per-cell-type donor coverage; explain parameter count, confounding, and the small-sample limitation.
C. Filter genes inside one cell type and fit one PyDESeq2 Old-versus-Young pilot; explain raw counts, dispersion, Wald statistic, and contrast direction.
D. Create a tidy per-cell-type results record with tested genes, up-in-Old, up-in-Young, and total DEG count at the stated threshold.
E. Create and interpret the sorted cell-type DEG-count plot; explain why it is not a direct score of biological aging.
F. Create one correctly labeled volcano plot for a selected top cell type; explain axes, threshold lines, colors, and safe gene labeling.
G. Create the second volcano plot and draft two evidence-bound figure captions.

First ask me to select A–G. Do not give code for later steps, a full loop, expected genes, or a biological conclusion.
```


# Section 3 — pathway analysis with GSEApy, KEGG, and Gene Ontology

## Goal

For the two cell types selected in Section 2, use **pre-ranked GSEA** to ask whether KEGG pathways and Gene Ontology Biological Process gene sets accumulate at the top or bottom of the complete Old-versus-Young DESeq2 ranking.

Use the full filtered DESeq2 result, ranked by the **signed Wald statistic**. Do not run pre-ranked GSEA only on the thresholded DEG list: that would turn a ranked enrichment question into a selected-gene analysis and discard weaker coordinated signal.

## Statistics and figures you must understand

- **Gene-set database:** record the exact current GSEApy/Enrichr library name, resource, species, and retrieval date. Verify the library name at runtime rather than guessing a versioned label.
- **ES and NES:** enrichment score is the running-sum signal; normalized enrichment score allows comparison across gene-set sizes and permutations. The sign follows the ranking direction that you define.
- **FDR q value:** multiple-testing-adjusted enrichment evidence across the tested gene sets. Declare your significance threshold before interpreting.
- **Leading-edge genes:** genes that contribute most to the enrichment signal. They are candidates for follow-up, not proof of a pathway mechanism.
- **Plots:** make a compact dot/bar plot of selected pathways and at least one enrichment curve per database for a pathway you interpret. Include NES, FDR q value, direction, cell type, and contrast in captions.

GSEApy supports current Enrichr library names or a GMT file. Use the package to list current human libraries, then select a KEGG library and a GO Biological Process library. Keep the two databases as separate runs so their multiple-testing universes are clear.


## Section 3 questions and required artifacts

**3A — Prepare the ranking.** For one top cell type, create a two-column ranked input from all tested genes: unique human gene symbol and signed DESeq2 Wald statistic. Check missing/duplicate symbols and state what a positive statistic means in your chosen Old-versus-Young contrast.

**3B — Confirm pathway resources.** Ask GSEApy for currently available human libraries. Select one KEGG resource and one GO Biological Process resource. Record exact library names and retrieval date; do not assume a library name from an old tutorial is still current.

**3C — Run and inspect pre-ranked GSEA.** Run the two resources separately with a fixed seed and documented gene-set size limits. Export results with term, ES, NES, nominal p value, FDR q value, and leading-edge genes. What question does this analysis answer that a volcano plot does not?

**3D — Communicate pathways.** For each top cell type, make an interpretable pathway plot and one enrichment curve from each database. Select terms by a declared FDR criterion and biological relevance—not solely by the largest NES. Name the leading-edge genes you checked in the DEG table.

**Deliverable:** KEGG and GO Biological Process GSEA result tables, two compact pathway figures for each selected cell type, and an evidence paragraph that names the database, ranking statistic, NES direction, FDR criterion, leading-edge genes, limitation, and next validation test.


### Section 3 staged AI prompt

```text
I am working on Section 3 of the Aging PBMC lab. Follow the AI coaching contract exactly.

My goal: for each of the two cell types selected from donor-level PyDESeq2 age-DE, run GSEApy pre-ranked GSEA separately for a current human KEGG library and a current human Gene Ontology Biological Process library. My ranking is the signed DESeq2 Wald statistic for Old versus Young, using all tested genes after filtering. I will record exact resource names, retrieval date, seed, gene-set size limits, ES, NES, FDR q value, and leading-edge genes.

Teach only the micro-step I choose:
A. Prepare and validate one ranked gene-statistic input; explain why all tested genes, not only DEGs, belong in pre-ranked GSEA.
B. Query GSEApy for current human library names and choose one KEGG and one GO Biological Process resource without guessing version labels.
C. Run one KEGG pre-ranked analysis with a fixed seed; inspect term, ES, NES, p value, FDR q value, and leading edge.
D. Run one GO Biological Process pre-ranked analysis separately and explain why its FDR is not pooled with KEGG.
E. Make one pathway summary plot and one enrichment curve; explain direction and leading-edge genes.
F. Audit a proposed pathway claim for overstatement and propose an orthogonal validation.

First ask me to select A–F. Do not provide full code for both cell types, both databases, or an answer key in one response.
```


# Final evidence memo and AI claim audit

Submit a brief report containing the following. Use your own observed results; do not borrow an AI-generated conclusion.

1. **Data and design:** list the eight donors, the age-group rule, labels used, raw-count source, biological replicate, and the limitations of the cross-sectional design.
2. **Section 1:** state the target-versus-rest definition; show method-comparison table(s), correlation plot(s), and Venn diagram(s). Explain why the methods can differ and which called more genes under your declared threshold.
3. **Section 2:** report filters, model formula, contrast direction, the cell-type DEG-count figure, and two volcano plots. State which two cell types had the most thresholded genes without calling them the “most aged” cell types.
4. **Section 3:** report KEGG and GO resource names, ranking statistic, FDR criterion, selected terms, NES direction, and verified leading-edge genes.
5. **Claim boundary:** write one observation, one inference, one limitation, and one specific next validation experiment or independent dataset test.

### Final AI claim-audit prompt

```text
Follow the AI coaching contract exactly. Do not rewrite my report or add findings.

Here are my evidence artifacts: donor/cell-type coverage table; Wilcoxon and pseudobulk method-comparison tables; signed-effect correlation figure; Venn figure; age-DE summary; two volcano plots; KEGG and GO pre-ranked GSEA tables/figures.

Here is my 4–6 sentence conclusion:
<<PASTE YOUR CONCLUSION>>

Separate direct observation, inference, limitation, and next validation test. Flag any use of cells as replicates, an ambiguous contrast direction, unsupported aging causality, a pathway-as-mechanism claim, missing library/version details, or unverified gene names. Then ask one multiple-choice revision question and wait.
```


In [ ]:
# Run only after completing your figures and memo.
mark_done("method_comparison", "aging_deg", "pathway_gsea", "evidence_memo")
print("Record your AI verification notes and submit only the artifacts requested by your instructor.")


## Sources and technical references

1. Zhu H, Chen J, Liu K, et al. *Human PBMC scRNA-seq–based aging clocks reveal ribosome to inflammation balance as a single-cell aging hallmark and super longevity.* **Science Advances** 2023; 9:eabq7599. https://doi.org/10.1126/sciadv.abq7599
2. GEO GSE213516, official count archive: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213516/suppl/GSE213516_RAW.tar
3. Han JD. *Cell type annotation for human PBMC scRNA-seq–based aging clocks.* Figshare, CC BY 4.0. https://doi.org/10.6084/m9.figshare.23789844.v1
4. Scanpy `read_10x_mtx` API: https://scanpy.readthedocs.io/en/stable/generated/scanpy.read_10x_mtx.html
5. Scanpy `rank_genes_groups` API and its warning about cell-level inflated p values: https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html
6. sc-best-practices, differential gene expression: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
7. decoupler pseudobulk workflow: https://decoupler.scverse.org/en/latest/notebooks/scell/rna_psbk.html
8. PyDESeq2 workflow: https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_minimal_pydeseq2_pipeline.html
9. GSEApy documentation: https://gseapy.readthedocs.io/en/latest/introduction.html

**Version note:** package versions are installed in the first Colab cell. Record the resolved versions printed by your own runtime in the report appendix.
